In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LightSource

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

# Baseline MD on the Müller–Brown Surface

### What this notebook does
- Loads and visualizes the MB potential (gnuplot layout: y varies fastest).
- Runs/visualizes plain Langevin MD (no bias) and its sampling density.
- Serves as the baseline before metadynamics/path-metadynamics.

### When to run (order)
1) From `day7_path_meta_dynamics`: `bash make_potential.sh` (once).
2) Run MD: `cd Exercise/0_md && ../plumed/pesmd < plumed.dat > md.log`.
3) Then execute this notebook top-to-bottom.

### Key notes
- `potential.dat` has a header + blank lines; we load with `comments='#'` and reshape column-major.
- Trajectory: `Exercise/0_md/colvar.out` (cv.x, cv.y).
- At T=0.5 K expect limited basin hopping—use this to motivate enhanced sampling.

### Quick questions
- How far does unbiased MD explore?
- Which parameter tweaks (T, friction, timestep) change basin visits without bias?
- Compare sampling map vs potential minima/saddles.


## 1. Load and Visualize the Potential Energy Surface

The potential was generated using a sum of Gaussian functions, creating a landscape with multiple basins and saddle points.

In [ ]:
# Load the potential energy surface
# Data stored column-major: for each x, all y are listed (gnuplot style)
potential_data = np.loadtxt('Exercise/potential.dat', comments='#')

# Extract unique x and y values
x_vals = np.unique(potential_data[:, 0])
y_vals = np.unique(potential_data[:, 1])

# Create meshgrid
X, Y = np.meshgrid(x_vals, y_vals)
# Reshape with y varying fastest, then transpose to match meshgrid
Z = potential_data[:, 2].reshape(len(x_vals), len(y_vals)).T

print(f"Potential grid size: {len(x_vals)} x {len(y_vals)}")
print(f"X range: [{x_vals.min():.2f}, {x_vals.max():.2f}]")
print(f"Y range: [{y_vals.min():.2f}, {y_vals.max():.2f}]")
print(f"Energy range: [{Z.min():.2f}, {Z.max():.2f}] K")

In [ ]:
# Create a beautiful visualization of the potential
fig, ax1 = plt.subplots(1, 1, figsize=(8, 7))

# 2D contour plot
levels = np.linspace(Z.min(), 0, 30)
contour = ax1.contourf(X, Y, Z, levels=levels, cmap='viridis')
ax1.contour(X, Y, Z, levels=levels, colors='white', alpha=0.3, linewidths=0.5)
cbar1 = plt.colorbar(contour, ax=ax1, label='Potential Energy (K)')
ax1.set_xlabel('cv.x')
ax1.set_ylabel('cv.y')
ax1.set_title('2D Potential Energy Surface (Contour)')
# ax1.set_xlim([-1.5, 1.5])
# ax1.set_ylim([-0.5, 2.5])
xmin = x_vals.min()
xmax = x_vals.max()
ymin = y_vals.min()
ymax = y_vals.max()
ax1.set_xlim([xmin, xmax])
ax1.set_ylim([ymin, ymax])
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Analyze the MD Trajectory

We ran a standard Langevin MD simulation at temperature T=0.5 K with friction coefficient γ=0.1. The simulation was started from position (0.6, 0.0).

### Simulation Parameters:
- **Temperature**: 0.5 K
- **Friction**: 0.1
- **Time step**: 0.001
- **Number of steps**: 500,000
- **Initial position**: (0.6, 0.0)

In [ ]:
# Load the MD trajectory
colvar = np.loadtxt('Exercise/0_md/colvar.out')
time = colvar[:, 0]
cv_x = colvar[:, 1]
cv_y = colvar[:, 2]

print(f"Trajectory length: {len(time)} frames")
print(f"Time range: {time[0]:.1f} - {time[-1]:.1f}")
print(f"Sampling frequency: every {int(time[1] - time[0])} steps")

In [ ]:
# Plot trajectory on the potential
fig, ax = plt.subplots(figsize=(10, 8))

# Plot potential as background
contour = ax.contourf(X, Y, Z, levels=levels, cmap='viridis', alpha=0.8)
ax.contour(X, Y, Z, levels=levels, colors='white', alpha=0.3, linewidths=0.5)

# Plot trajectory
scatter = ax.scatter(cv_x, cv_y, c=time, s=1, cmap='hot', alpha=0.6, zorder=10)
ax.plot(cv_x[0], cv_y[0], 'go', markersize=15, label='Start', zorder=20)
ax.plot(cv_x[-1], cv_y[-1], 'r*', markersize=20, label='End', zorder=20)

cbar1 = plt.colorbar(contour, ax=ax, label='Potential Energy (K)', pad=0.12)
cbar2 = plt.colorbar(scatter, ax=ax, label='Time (steps)', fraction=0.046, pad=0.04)

ax.set_xlabel('cv.x')
ax.set_ylabel('cv.y')
ax.set_title('MD Trajectory on 2D Potential Energy Surface')
ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-0.5, 2.5])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Statistical Analysis

Let's analyze how well the MD simulation samples different regions of the potential energy surface.

In [ ]:
# Plot time series of collective variables
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(time, cv_x, linewidth=0.5)
ax1.set_ylabel('cv.x')
ax1.set_title('Collective Variable Evolution')
ax1.grid(True, alpha=0.3)

ax2.plot(time, cv_y, linewidth=0.5, color='orange')
ax2.set_xlabel('Time (steps)')
ax2.set_ylabel('cv.y')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create 2D histogram to show sampling density
fig, ax = plt.subplots(figsize=(6, 5))

# Create 2D histogram
H, xedges, yedges = np.histogram2d(cv_x, cv_y, bins=50, 
                                    range=[[-1.5, 1.5], [-0.5, 2.5]])

# Plot potential as contours
contour = ax.contour(X, Y, Z, levels=20, colors='white', alpha=0.5, linewidths=1)

# Plot sampling density
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
im = ax.imshow(H.T, origin='lower', extent=extent, cmap='hot', 
               interpolation='gaussian', alpha=0.8, aspect='auto')

cbar = plt.colorbar(im, ax=ax, label='Sampling Density')
ax.set_xlabel('cv.x')
ax.set_ylabel('cv.y')
ax.set_title('Sampling Density from MD Simulation')
ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-0.5, 2.5])

plt.tight_layout()
plt.show()

## Conclusions

From this analysis, we observe:

1. **Limited Sampling**: The MD trajectory tends to stay in the initial energy basin, showing limited exploration of the full potential energy surface.

2. **Rare Event Problem**: The energy barriers prevent the system from spontaneously transitioning between different minima on the MD timescale.

3. **Need for Enhanced Sampling**: Standard MD at this temperature is insufficient for exploring all relevant regions of the potential. This motivates the use of enhanced sampling methods like metadynamics.

In the next notebooks, we'll explore how **metadynamics** and **path metadynamics** can overcome these limitations and efficiently sample the entire potential energy surface.